# Rule-based scheduler demo

This notebook demonstrates Phase 1 cold-start booking rules: clinic hours, slot capacity, queue-aware recommendations, and staff-recorded outcomes.

In [ ]:
import json
import sys
from datetime import date, datetime, time, timedelta
from pathlib import Path

import pandas as pd

project_root = Path.cwd()
if not (project_root / "app").is_dir():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from app.db import get_connection, init_db, save_clinic_config
from app.schemas import ArrivalEvent, BookingRequest, ClinicConfig, ConsultationEvent
from app.scheduler import (
    ClinicClosedError,
    DuplicateBookingError,
    book_appointment,
    classify_wait,
    day_abbrev,
    generate_slots,
    record_arrival,
    record_consultation,
)

sample_path = project_root / "data" / "sample_clinic_config.json"
config = ClinicConfig.model_validate_json(sample_path.read_text(encoding="utf-8"))
demo_db_path = str(project_root / "data" / "demo.sqlite")
init_db(path=demo_db_path)
save_clinic_config(config, path=demo_db_path)

demo_date = date.today() + timedelta(days=1)
while day_abbrev(demo_date) not in config.available_days:
    demo_date += timedelta(days=1)
connection = get_connection(demo_db_path)
try:
    connection.execute(
        "DELETE FROM appointments WHERE appointment_date = ?",
        (demo_date.isoformat(),),
    )
    connection.commit()
finally:
    connection.close()

print(f"Demo database: {demo_db_path}")
print(f"Next available clinic day: {demo_date} ({day_abbrev(demo_date)})")

## Slots for the next available day

Slots advance by appointment duration plus buffer. Starts whose consultation duration overlaps the configured break are skipped.

In [ ]:
slots = generate_slots(config, demo_date)
pd.DataFrame({"slot_start": slots})

## Book eight patients

The rule-based estimate uses patients already sharing a slot, plus a small peak-period adjustment. Queue progression moves later patients to later available slots.

In [ ]:
responses = []
for index in range(8):
    request = BookingRequest(
        patient_name=f"Demo Patient {index + 1}",
        patient_phone=f"+962-7-9000-{index:04d}",
        appointment_date=demo_date,
    )
    responses.append(book_appointment(request, path=demo_db_path))

response_rows = [
    {
        **response.model_dump(mode="json"),
        "classify_wait": classify_wait(response.estimated_wait_min, config),
    }
    for response in responses
]
pd.DataFrame(response_rows)

In [ ]:
for response in responses:
    print(
        f"Queue {response.queue_number}: wait={response.estimated_wait_min} min, "
        f"consultation={response.expected_consultation_time}, "
        f"arrive={response.recommended_arrival_time}"
    )

## Expected booking errors

The scheduler reports a closed day and a repeated active phone/date booking explicitly.

In [ ]:
closed_date = demo_date + timedelta(days=1)
while day_abbrev(closed_date) in config.available_days:
    closed_date += timedelta(days=1)
try:
    book_appointment(
        BookingRequest(
            patient_name="Closed Day Patient",
            patient_phone="+962-7-9111-0000",
            appointment_date=closed_date,
        ),
        path=demo_db_path,
    )
except ClinicClosedError as exc:
    print(f"Closed-day booking rejected: {exc}")

try:
    book_appointment(
        BookingRequest(
            patient_name="Duplicate Patient",
            patient_phone="+962-7-9000-0000",
            appointment_date=demo_date,
        ),
        path=demo_db_path,
    )
except DuplicateBookingError as exc:
    print(f"Duplicate booking rejected: {exc}")

## Record staff arrival and consultation events

Actual waiting time is measured from the recorded arrival to consultation start. Consultation duration and completed outcome are saved for each patient.

In [ ]:
completed_records = []
for response in responses[:2]:
    scheduled = datetime.combine(response.appointment_date, response.expected_consultation_time)
    actual_arrival = scheduled + timedelta(minutes=4)
    consult_start = actual_arrival + timedelta(minutes=11)
    consult_end = consult_start + timedelta(minutes=18)

    record_arrival(
        ArrivalEvent(
            appointment_id=response.appointment_id,
            actual_arrival=actual_arrival,
        ),
        path=demo_db_path,
    )
    completed_records.append(
        record_consultation(
            ConsultationEvent(
                appointment_id=response.appointment_id,
                start=consult_start,
                end=consult_end,
            ),
            path=demo_db_path,
        )
    )

pd.DataFrame([record.model_dump(mode="json") for record in completed_records])

## From booking logs to Phase 2 data

Each appointment now retains the booking-time conditions and rule-based prediction alongside staff-recorded arrival, lateness, consultation duration, actual wait, and outcome. Once enough representative records have been collected and checked, those labeled observations can form a Phase 2 dataset for evaluating a future wait-time model. The current scheduler remains rules-only.